<a href="https://colab.research.google.com/github/shweta-shankar/MAS-AI-Research-Assistant/blob/gitSearch/gitSearch_with_Domain_Restriction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pdfplumber groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 39.4 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from groq import Groq
import pdfplumber
import time

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

In [3]:
from google.colab import files

print("Upload your gitingest txt file")
uploaded = files.upload()
gitingest_file = list(uploaded.keys())[0]

#print("Upload paper 1")
#uploaded = files.upload()
#pdf1 = list(uploaded.keys())[0]

#print("Upload paper 2")
#uploaded = files.upload()
#pdf2 = list(uploaded.keys())[0]

Upload your gitingest txt file


Saving goellab-digibone-8a5edab282632443.txt to goellab-digibone-8a5edab282632443.txt


In [4]:
def load_pdf(pdf_path):
    lines = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text:
                lines.append(f"================================================\n")
                lines.append(f"FILE: {pdf_path} (page {i+1})\n")
                lines.append(f"================================================\n")
                for line in text.splitlines(keepends=True):
                    lines.append(line)
    return lines

with open(gitingest_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

#lines += load_pdf(pdf1)
#lines += load_pdf(pdf2)

print(f"Total lines loaded: {len(lines)}")

Total lines loaded: 988


In [6]:
def search(query, lines):
    results = []
    for i, line in enumerate(lines):
        if query.lower().replace(" ", "").replace("-", "") in line.lower().replace(" ", "").replace("-", ""):
            current_file = "unknown"
            for j in range(i, 0, -1):
                if lines[j].startswith("FILE:"):
                    current_file = lines[j].replace("FILE:", "").strip()
                    break
            start = max(0, i - 2)
            end = min(len(lines), i + 20)
            snippet = "".join(lines[start:end])
            results.append({
                "file": current_file,
                "line_number": i + 1,
                "snippet": snippet
            })
    return results


def call_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


def extract_words(question):
    prompt = f"""Extract 3-5 keywords strictly from the question below.
Only use words that actually appear in the question.
Do NOT add related terms or your own knowledge.
Return ONLY a comma separated list, nothing else.

Question: {question}
"""
    return call_llm(prompt).split(",")


def filter_results(results, question):
    question_words = set(question.lower().replace("-","").replace(" ","").split())
    scored = []
    for r in results:
        snippet_words = set(r["snippet"].lower().replace("-","").replace(" ","").split())
        score = len(question_words & snippet_words)
        scored.append((score, r))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [r for score, r in scored[:8]]


In [7]:
def is_in_scope(question):
    prompt = f"""You are a domain checker for a pediatric bone age assessment system called digiBONE.
Answer ONLY with 'YES' or 'NO'. No explanation.

Is this question related to any of these topics:
- Pediatric bone age assessment
- Hand X-rays or skeletal maturity
- The digiBONE system or its code
- Segmentation of hand X-ray images
- Greulich-Pyle method
- CNN or deep learning models for bone age
- Indian pediatric datasets (RSNA, HCJBA)

Question: {question}
"""
    answer = call_llm(prompt).strip().upper()
    return "YES" in answer


def ask_llm(question, search_results):
    context = ""
    for r in search_results:
        context += f"\nFile: {r['file']} | Line: {r['line_number']}\n{r['snippet']}\n---"

    prompt = f"""You are a helpful assistant for the digiBONE pediatric bone age assessment system.
You are ONLY allowed to answer questions about pediatric bone age assessment, hand X-rays,
skeletal maturity, segmentation, CNN models, and the digiBONE system.

If the question is outside this domain, respond with exactly:
"This question is outside the scope of this system."

Using ONLY the context below, write a clear and well structured answer.
- Start with the plain English definition or explanation first
- Then add implementation details and how it works in this specific project
- End every claim with a citation like [README.md:42]
- Do NOT start with code or file references
- If the context doesn't contain the answer, say so explicitly

Context:
{context}

Question: {question}
"""
    return call_llm(prompt)


def run_query(question):
    start = time.time()

    # ── Method 1: Gate the question ───────────────────────────────
    print("Checking if question is in scope...")
    if not is_in_scope(question):
        print("This question is outside the scope of this system.")
        return

    keywords = extract_words(question)
    print("Keywords:", keywords)

    results = []
    for keyword in keywords:
        results += search(keyword, lines)

    # ── Method 3: Check if any results found ──────────────────────
    if len(results) == 0:
        print("No relevant information found. This question may be outside the scope of this system.")
        return

    # deduplicate
    seen = set()
    unique_results = []
    for r in results:
        key = (r["file"], r["line_number"])
        if key not in seen:
            seen.add(key)
            unique_results.append(r)

    print("Unique results:", len(unique_results))

    filtered = filter_results(unique_results, question)
    answer = ask_llm(question, filtered)  # ← Method 2 is inside here

    end = time.time()
    print(f"\nAnswer:\n{answer}")
    print(f"\nTime taken: {(end-start)/60:.2f} mins")

In [8]:
run_query("can this system be used to estimate the bone age of mummies?")

Checking if question is in scope...
This question is outside the scope of this system.


In [9]:
run_query("what is the basketball match score")

Checking if question is in scope...
This question is outside the scope of this system.


In [10]:
run_query("What MAD values did the transfer-learned full-hand model achieve on the HCJBA dataset?")

Checking if question is in scope...
Keywords: ['MAD', ' transfer-learned', ' full-hand', ' HCJBA']
Unique results: 22

Answer:
The Mean Absolute Deviation (MAD) value is a measure used to evaluate the performance of a model by calculating the average difference between predicted and actual values. In the context of bone age assessment, MAD values represent the average difference between the predicted bone age and the actual bone age of a child. 

The digiBONE system, which utilizes transfer learning for its full-hand model, has reported MAD values for the full-hand model on the HCJBA dataset. According to Table 1 in the README.md file, the transfer-learned full-hand model achieved MAD values of 5.7 months for males and 5.9 months for females [README.md:55]. 

These values indicate that, on average, the predicted bone age by the full-hand model differed from the actual bone age by approximately 5.7 months for males and 5.9 months for females [README.md:55].

Time taken: 0.03 mins


In [11]:
run_query("How were the segmental GP labels assigned to each image?")

Checking if question is in scope...
Keywords: ['GP', ' labels', ' image', ' segmental']
Unique results: 105

Answer:
The assignment of segmental GP labels to each image is based on the Segmental Greulich-Pyle (SGP) method, which divides the full hand into three meaningful anatomical segments: Shortbones (Metacarpal and Phalanges), Carpals, and Wrist (Radius and Ulna) [README.md:15]. 

In the implementation, two separate segmentation models with a Unet backbone were trained to segment the carpals and the wrist region of the X-ray images [README.md:31]. These models were used to generate the carpal and wrist crops of Indian X-ray images after appropriate mask post-processing, and the full-hand image was then used to obtain the shortbones crop [README.md:31]. 

However, the exact process of assigning segmental GP labels to each image is not explicitly stated in the provided context. It can be inferred that the bone age prediction models for each segment were used to determine the segmenta

In [12]:
run_query("How are short bone masks preprocessed to generate short-bone segments?")

Checking if question is in scope...
Keywords: ['bone', ' masks', ' preprocessed', ' short', ' segments']
Unique results: 101

Answer:
Short bone masks are preprocessed to generate short-bone segments through a post-processing step that involves subtracting the carpal and wrist crops from the full-hand image [README.md:28]. This process is implemented in the `segment_crops.py` script, which is used to generate the carpal and wrist crops of Indian X-ray images after appropriate mask post-processing [README.md:28]. The resulting short-bone segments can then be used for further analysis, such as bone age assessment [README.md:28].

Time taken: 0.02 mins


In [13]:
run_query("Why is the bone-age assessment for boys and girls different?")

Checking if question is in scope...
Keywords: ['bone-age', ' assessment', ' boys', ' girls', ' different']
Unique results: 53

Answer:
Bone-age assessment for boys and girls can be different because the maturation of bones in the hand is influenced by different hormonal factors. For instance, sex hormones govern the maturation of tubular bones, such as metacarpal, phalanges, radius, and ulna, but carpal maturation follows a different pattern [2]. This difference in bone maturation rates between boys and girls can lead to variations in bone age assessment. As a result, the digiBONE system uses separate bone age prediction models for males and females to account for these differences [4]. The system's use of separate models for each sex allows for more accurate bone age assessments, taking into account the distinct hormonal influences on bone development in boys and girls [README.md:42].

Time taken: 0.02 mins


In [14]:
run_query("Is the bone-age assessment for prepubesent boys and girls different?")

Checking if question is in scope...
Keywords: ['bone-age', ' assessment', ' prepubesent', ' boys', ' girls']
Unique results: 52

Answer:
Bone age assessment can vary between prepubescent boys and girls due to differences in hormonal influences on bone maturation. The Greulich and Pyle (GP) method, a widely used technique for bone age assessment, takes into account the differences in bone maturation between the sexes [1]. However, the provided context does not contain specific information on how the bone-age assessment differs between prepubescent boys and girls. It only mentions that the boneage prediction models were trained separately for male and female on the RSNA dataset [4]. Therefore, while it is known that sex hormones govern the maturation of bones differently, the exact differences in bone-age assessment between prepubescent boys and girls are not explicitly stated in the given context [2].

Time taken: 0.02 mins


In [15]:
run_query("Is the bone-age assessment associated with diabeties")

Checking if question is in scope...
This question is outside the scope of this system.


In [16]:
run_query("Should a child take calcium supplements to improve or match the chronical bone-age associated with their specific age?")

Checking if question is in scope...
This question is outside the scope of this system.
